# BSL Word Recognition — complete reproduction, one notebook, one run

**When this notebook is run top-to-bottom on the `bsl` kernel, every artefact of the project
is produced afresh during execution**: MediaPipe landmarks for all 393 shipped videos, the
verified data splits, the EDA, the sanity checks, all 32 training runs (v1 grid, lr/wd
selection grid, v2 grid), the validation summaries, the reproduction of the recorded
one-shot held-out test results, the latency benchmark, every dissertation figure and
table, and the unit-test suite. **Nothing is read from precomputed results — the archive
ships without any.**

| Part | Contents | Approx. time (laptop GPU) |
|------|----------|--------------------------|
| A | Setup, data inventory, landmark extraction (293 clips), preprocessing | ~10 min |
| B | Exploratory data analysis + split verification | ~2 min |
| C | Sanity checks, v1 12-run grid, lr/wd grid, validation evaluation, benchmark | ~25 min |
| D | v2 partition, v2 12-run grid, one-shot test reproduction (both test sets) | ~15 min |
| E | Dissertation figures and tables | ~2 min |
| F | Unit-test suite, final recap | ~1 min |

Notes for the examiner:

- **No network access is needed or used.** All video data ships in this archive
  (see `DATA_NOTICE.txt`); the polite download tooling (`src/download.py`,
  `src/download_signbank.py`) is included for provenance but never invoked here.
- **One-shot discipline.** The held-out test sets were formally evaluated exactly once,
  on 2026-07-27, with model selection frozen beforehand. Part D
  *reproduces* those recorded evaluations from a fresh retrain with identical seeds; it
  does not make any new selection or tuning decision.
- **Determinism.** Every run is seeded and cuDNN-deterministic, so on the original
  machine and library versions (per-run `env.txt`) numbers reproduce exactly. On
  different hardware, training results may shift by a small amount; the recorded values
  are printed alongside every reproduced number for comparison. The v1 grid's
  dissertation-era records were additionally trained on CPU under torch 2.3.1, so fresh
  GPU numbers for Part C may differ slightly from the recorded 2026-07-09 entries.
- Environment: the `bsl` conda env (`python 3.11`, `pip install -r requirements.txt`,
  mediapipe 0.10.9 required for extraction). A CUDA build of torch is strongly
  recommended — CPU-only, the training parts take several hours.


In [ ]:
import os
import sys
from pathlib import Path

# Walk up from the notebook's directory until the repo root (the folder with src/) is found.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not find the repository root (no src/ directory above cwd)")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("working directory:", Path.cwd())


In [ ]:
import json

# torch is imported before pandas/scipy/sklearn on purpose: with a CUDA torch build on
# Windows, importing it after those packages can fail DLL initialisation (WinError 1114).
import torch

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("torch  :", torch.__version__, "| cuda available:", torch.cuda.is_available())


In [ ]:
# Data inventory: everything needed ships in this archive; nothing is precomputed.
VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
n_raw = sum(1 for p in Path("data/raw_videos").rglob("*") if p.suffix.lower() in VIDEO_EXT)
n_sb = sum(1 for p in Path("data/test_videos_signbank").rglob("*") if p.suffix.lower() in VIDEO_EXT)
print(f"training/val/test source videos : {n_raw} (expected 293)")
print(f"SignBank cross-corpus videos    : {n_sb} (expected 100)")
for f in ["data/metadata.csv", "data/vocabulary.csv", "data/splits.json",
          "data/splits_v2.json", "data/signbank_gloss_map.csv",
          "data/signbank_gloss_map_curated.csv", "data/signbank_metadata.csv",
          "DATA_NOTICE.txt"]:
    print(("OK      " if Path(f).is_file() else "MISSING ") + f)
pre = [p for p in ("data/landmarks", "results/runs", "results/runs_v2") if Path(p).exists()]
if pre:
    print(f"\nNOTE: existing outputs found ({', '.join(pre)}) - idempotent cells may reuse them.")
else:
    print("\nClean tree: no landmarks or results exist yet - everything below is produced live.")
assert n_raw == 293 and n_sb == 100, "video inventory does not match the shipped counts"


# Part A — From shipped videos to model-ready features

## 1 — Check availability and download clips

CLI: `python -m src.download --dry-run` then `python -m src.download`

`src/download.py` is a **polite scraper** for SignBSL.com: one identifying user agent, at
least 1 second between HTTP requests, and it is **idempotent** — any source URL already in
`data/metadata.csv` is skipped, so re-running it never re-downloads anything. Videos land in
`data/raw_videos/{word}/{clip_id}.mp4` and one metadata row is appended (and persisted)
per clip. Downloaded videos are for local research use only and are never redistributed.

`main()` accepts an argv list, so it is invoked exactly as the CLI invokes it. A dry run prints
per-word variant counts and writes nothing.

In [ ]:
vocabulary = pd.read_csv("data/vocabulary.csv")
print(f"{len(vocabulary)} vocabulary words")
vocabulary.head()


In [ ]:
# The real download of the full vocabulary (idempotent: existing clips are skipped).
# Takes a while on first run because of the polite 1 request/second limit.
RUN_DOWNLOAD = False

if RUN_DOWNLOAD:
    download_main([])  # same defaults as `python -m src.download`
else:
    n_rows = len(pd.read_csv("data/metadata.csv"))
    print(f"skipped (RUN_DOWNLOAD = False) — metadata.csv already has {n_rows} clips")


## 2 — Extract landmarks

CLI: `python -m src.extract --videos data/raw_videos --out data/landmarks`

Each video is run through **MediaPipe Holistic 0.10.9** (legacy Solutions API,
`model_complexity=1`, one fresh `Holistic` per clip so tracking state never leaks between
clips). Per frame the fixed **105-landmark subset** (33 pose + 21 per hand +
30 mouth points) is retained, with NaN where a block was not detected, plus a per-frame presence flag
for left hand / right hand / face. Results are cached as compressed
`data/landmarks/{word}/{clip_id}.npz`, and the extraction is **idempotent** — clips with an
existing `.npz` are skipped, so this cell is cheap to re-run.

Afterwards it prints the mean **hand-dropout rate per source organisation** (fraction of
frames where a hand is missing while the body is visible) — a data-quality signal that also
feeds the EDA notebook.

In [ ]:
from src.extract import main as extract_main

# Idempotent: already-cached clips are skipped. Requires mediapipe==0.10.9 + opencv.
extract_main(["--videos", "data/raw_videos", "--out", "data/landmarks"])


## 3 — Inspect a cached clip: the 105-landmark layout

`src/landmarks.py` fixes the layout of every frame: a `(105, 3)` array of `(x, y, z)` in
block order **pose `[0:33]` → left hand `[33:54]` → right hand `[54:75]` → mouth `[75:105]`**.
Only the mouth region of the 468-point face mesh is kept, because mouthings carry
discriminative information in BSL while the full mesh would grow the feature size roughly five-fold.
Flattened, a frame is **315 features** (`FEATURES_PER_FRAME`).

In [ ]:
from src import landmarks as L
from src.extract import hand_dropout_rate

print(f"landmarks per frame : {L.N_LANDMARKS}  "
      f"(pose {L.POSE_N} + hands 2x{L.HAND_N} + mouth {L.MOUTH_N})")
print(f"features per frame  : {L.FEATURES_PER_FRAME}  ({L.N_LANDMARKS} x {L.N_COORDS})")

sample_npz = sorted(Path("data/landmarks").glob("*/*.npz"))[0]
with np.load(sample_npz) as d:
    raw = d["landmarks"].astype(np.float32)
    presence = d["presence"].astype(np.float32)
    fps = float(d["fps"])

print(f"\nsample clip         : {sample_npz}")
print(f"landmarks shape     : {raw.shape}   (T frames, 105 landmarks, xyz)")
print(f"presence shape      : {presence.shape}  (per-frame [left hand, right hand, face])")
print(f"fps                 : {fps:.2f}  ->  {raw.shape[0] / fps:.2f} s")
print(f"detection rates     : left hand {presence[:, 0].mean():.0%}, "
      f"right hand {presence[:, 1].mean():.0%}, face {presence[:, 2].mean():.0%}")
print(f"hand dropout rate   : {hand_dropout_rate(raw, presence):.3f}")


## 4 — Preprocessing: gap fill → normalise → NaN→0 → resample

This is the exact per-clip pipeline the dataset applies before a model ever sees a clip:

1. **`fill_gaps`** (`src/extract.py`) — linearly interpolates missing hand/mouth runs of
   ≤ 5 frames that are bounded by detections on both sides; longer or leading/trailing gaps
   stay NaN.
2. **`normalise_sequence`** (`src/normalise.py`) — per frame, shifts the skeleton so the
   shoulder midpoint is the origin and divides by the x-y shoulder distance. This makes the
   representation invariant to where the signer stands and to camera zoom.
3. **NaN → 0** — anything still missing becomes zero.
4. **`resample_sequence`** (`src/dataset.py`) — linear temporal resampling to a fixed
   **64 frames**, giving the `64 × 315` matrix the models consume.

`load_clip` bundles steps 1–3 (the resample happens inside the dataset, after optional
temporal augmentation).

In [ ]:
from src.extract import fill_gaps
from src.normalise import normalise_sequence
from src.dataset import load_clip, resample_sequence

filled = fill_gaps(raw, presence)               # 1. interpolate short gaps (<= 5 frames)
normed = normalise_sequence(filled)             # 2. shoulder-centred, shoulder-scaled
clean = np.nan_to_num(normed, nan=0.0)          # 3. remaining NaN -> 0
fixed = resample_sequence(clean, 64)            # 4. linear resample to 64 frames

print(f"NaN values: raw {np.isnan(raw).sum()} -> gap-filled {np.isnan(filled).sum()} "
      f"-> after NaN->0: {np.isnan(clean).sum()}")
print(f"shape: {raw.shape} -> {fixed.shape}")

# load_clip == steps 1-3 in one call
assert np.allclose(load_clip(sample_npz), clean, atol=1e-6)
print("load_clip matches the manual steps 1-3")


In [ ]:
# Invariance check (the property the unit tests assert): translating the whole skeleton
# and uniformly rescaling it must not change the normalised output.
rng = np.random.default_rng(0)
shift = rng.uniform(-0.5, 0.5, size=3).astype(np.float32)
scale = np.float32(1.7)

a = np.nan_to_num(normalise_sequence(filled), nan=0.0)
b = np.nan_to_num(normalise_sequence(filled * scale + shift), nan=0.0)
diff = np.abs(a - b).max()
print(f"max |difference| after global shift + rescale of the input: {diff:.2e}")
assert diff < 1e-3


In [ ]:
# Skeleton before vs after normalisation. Matplotlib skips NaN points automatically,
# so undetected landmarks simply do not appear in the raw plot.
POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (15, 17), (15, 19), (15, 21), (16, 18), (16, 20), (16, 22),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (24, 26), (26, 28),
    (27, 29), (29, 31), (28, 30), (30, 32),
]


def plot_skeleton(ax, frame, title):
    pose = frame[L.POSE_SLICE]
    for i, j in POSE_CONNECTIONS:
        ax.plot([pose[i, 0], pose[j, 0]], [pose[i, 1], pose[j, 1]],
                color="tab:blue", lw=1.5, zorder=2)
    ax.scatter(frame[:, 0], frame[:, 1], s=4, color="tab:red", zorder=3)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.invert_yaxis()  # image convention: y grows downwards


mid = raw.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_skeleton(axes[0], raw[mid], f"raw, image coordinates (frame {mid})")
plot_skeleton(axes[1], fixed[32], "normalised, shoulder units (resampled frame 32)")
fig.suptitle(sample_npz.stem)
plt.show()


# Part B — Exploratory data analysis

*(merged from `notebooks/eda.ipynb`; the augmentation visual lives in Part A's pipeline section below)*

# Exploratory Data Analysis — BSL Word Transformer

Answers the handbook's five EDA questions from `data/metadata.csv` and the cached
landmark `.npz` files, then creates the organisation-grouped train/val split.

1. How many clips per word (class balance)?
2. How long are clips relative to the fixed 64-frame model input?
3. How often do hands drop out of MediaPipe tracking, per source?
4. What do the extracted skeletons actually look like?
5. How are clips distributed across source organisations (the split grouping unit)?

Requires `python -m src.download`, `python -m src.extract` to have been run
(see README workflow steps 1-3). Cells raise a clear `FileNotFoundError` otherwise.

In [ ]:
from pathlib import Path
import sys
import json

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = REPO / 'data'
META_CSV = DATA / 'metadata.csv'
LANDMARKS = DATA / 'landmarks'

if not META_CSV.exists():
    raise FileNotFoundError(
        f'{META_CSV} not found - run `python -m src.download` first (README workflow step 2).')

meta = pd.read_csv(META_CSV)
print(f'{len(meta)} clips, {meta["word"].nunique()} words, '
      f'{meta["organisation"].nunique()} organisations')
meta.head()

## Q1 — Clips per word (class balance)

In [ ]:
counts = meta['word'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(14, 4))
counts.plot.bar(ax=ax, color='steelblue')
ax.axhline(counts.mean(), color='grey', ls='--', label=f'mean = {counts.mean():.1f}')
ax.set_ylabel('clips')
ax.set_title('Clips per word')
ax.legend()
plt.tight_layout()
plt.show()
counts.describe()

## Q2 — Clip duration distribution vs the fixed 64-frame input

Frame counts are `duration_s * fps`. The red line marks the 64-frame budget every
clip is linearly resampled to; clips far above it lose temporal detail, clips
below it are stretched.

In [ ]:
frames = (meta['duration_s'] * meta['fps']).dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(frames, bins=40, color='steelblue', edgecolor='white')
ax.axvline(64, color='crimson', ls='--', label='64-frame model input')
ax.set_xlabel('frames per clip')
ax.set_ylabel('count')
ax.set_title('Clip length distribution vs fixed 64-frame resampling')
ax.legend()
plt.tight_layout()
plt.show()
meta['duration_s'].describe()

## Q3 — Hand/face dropout rate per source

Fraction of frames where MediaPipe failed to detect each block, from the cached
`presence` arrays (columns: left hand, right hand, face). High hand dropout is
the main data-quality risk for a pose-based pipeline.

In [ ]:
if not LANDMARKS.exists():
    raise FileNotFoundError(
        f'{LANDMARKS} not found - run '
        '`python -m src.extract --videos data/raw_videos --out data/landmarks` first.')

rows = []
for _, r in meta.iterrows():
    npz_path = LANDMARKS / r['word'] / f"{r['clip_id']}.npz"
    if not npz_path.exists():
        continue
    presence = np.load(npz_path)['presence']  # (T, 3): left hand, right hand, face
    rows.append({
        'source': r['source'],
        'clip_id': r['clip_id'],
        'left_hand_dropout': 1.0 - presence[:, 0].mean(),
        'right_hand_dropout': 1.0 - presence[:, 1].mean(),
        'face_dropout': 1.0 - presence[:, 2].mean(),
    })
drop = pd.DataFrame(rows)
print(f'presence loaded for {len(drop)} cached clips')
drop.groupby('source')[['left_hand_dropout', 'right_hand_dropout',
                        'face_dropout']].mean().round(3)

## Q4 — Skeleton overlays for sample clips

Mid-clip frames in raw MediaPipe image coordinates (y-axis inverted so figures are
upright). Blocks: pose (blue), left hand (orange), right hand (green), mouth (red).

In [ ]:
from src.landmarks import POSE_SLICE, LEFT_HAND_SLICE, RIGHT_HAND_SLICE, MOUTH_SLICE

sample = meta.groupby('word').head(1).sample(
    n=min(6, meta['word'].nunique()), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    npz_path = LANDMARKS / r['word'] / f"{r['clip_id']}.npz"
    if not npz_path.exists():
        ax.set_title(f"{r['clip_id']} (no cache)")
        ax.axis('off')
        continue
    seq = np.load(npz_path)['landmarks']  # (T, 105, 3), raw image coords
    mid = seq[len(seq) // 2]
    for sl, colour in [(POSE_SLICE, 'tab:blue'), (LEFT_HAND_SLICE, 'tab:orange'),
                       (RIGHT_HAND_SLICE, 'tab:green'), (MOUTH_SLICE, 'tab:red')]:
        ax.scatter(mid[sl, 0], mid[sl, 1], s=8, c=colour)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(r['word'], fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('Mid-clip landmark frames (pose / left hand / right hand / mouth)')
plt.tight_layout()
plt.show()

## Q5 — Source organisation distribution

Splits are grouped by organisation so no organisation (and hence, plausibly, no
signer or studio style) appears in both train and val.

In [ ]:
org = (meta.groupby('organisation')
           .agg(clips=('clip_id', 'count'), words=('word', 'nunique'))
           .sort_values('clips', ascending=False))
fig, ax = plt.subplots(figsize=(8, 3))
org['clips'].plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('clips')
ax.set_title('Clips per source organisation (split grouping unit)')
plt.tight_layout()
plt.show()
org

## Create train/val splits

Greedy organisation-grouped assignment targeting ~80/20 by clip count
(`src.dataset.make_splits`); writes `data/splits.json` and prints a summary.

In [ ]:
# The committed data/splits.json was built with EXPLICIT validation organisations
# (chosen offline by exhaustive subset enumeration - recorded 2026-07-09), so
# rather than re-running the greedy search the split is regenerated with those
# organisations and verified to match the committed record exactly.
from src.dataset import make_splits, load_splits

committed = load_splits("data/splits.json")
regen = make_splits(str(META_CSV), str(DATA / "vocabulary.csv"),
                    val_frac=0.2, seed=42, val_orgs=committed["val_orgs"])
assert regen["train"] == committed["train"] and regen["val"] == committed["val"], \
    "regenerated split differs from the committed data/splits.json!"
splits = committed
n_train, n_val = len(splits["train"]), len(splits["val"])
print("VERIFIED: regenerated split matches the committed data/splits.json exactly")
print(f"train {n_train} / val {n_val} clips ({n_val / (n_train + n_val):.1%} val); "
      f"val orgs: {', '.join(splits['val_orgs'])}; {len(splits['label_list'])} classes")


## 6 — `BSLDataset`: from clip ids to training batches

`BSLDataset` (`src/dataset.py`) wraps sections 4 and 7: per item it runs
`load_clip` → *(temporal augmentation if enabled)* → 64-frame resample →
*(spatial augmentation if enabled)* → flatten to a `(64, 315)` float32 tensor.
Class index = position of the clip's word in `label_list`. Every DataLoader uses `num_workers=0`, and the shuffling train loader draws from a seeded `torch.Generator` — part of the determinism guarantee.

In [ ]:
from torch.utils.data import DataLoader

from src.dataset import BSLDataset, clip_id_to_word

label_list = splits["label_list"]
label_to_idx = {word: i for i, word in enumerate(label_list)}

train_ids = splits["train"]
train_labels = [label_to_idx[clip_id_to_word(c)] for c in train_ids]

train_ds = BSLDataset(train_ids, train_labels, "data/landmarks",
                      augment=True, seq_len=64, seed=42)

features, target = train_ds[0]
print(f"one item : features {tuple(features.shape)} {features.dtype}, "
      f"label {target} ({label_list[target]})")

generator = torch.Generator()
generator.manual_seed(42)
loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0,
                    generator=generator)
xb, yb = next(iter(loader))
print(f"one batch: inputs {tuple(xb.shape)}, targets {tuple(yb.shape)}")


## 7 — Augmentation + visual sanity check

Script: `python scripts/plot_augmentations.py` (writes `results/figures/augmentation_check.png`)

`src/augment.py` provides label-preserving perturbations, each firing independently with
p = 0.5:

- **spatial** (after the 64-frame resample): rotation ±15° in the x-y plane, uniform scale
  0.9–1.1, x-y translation ±0.05, Gaussian jitter σ = 0.01;
- **temporal** (before the resample): speed change ×0.8–1.2 **or** dropping up to 10% of
  frames, never going below 8 frames.

The point of the visual check is to confirm the transforms are big enough to matter but
small enough that the sign stays the same sign.

In [ ]:
from src.augment import spatial_augment, temporal_augment

aug_rng = np.random.default_rng(7)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
plot_skeleton(axes[0], fixed[32], "original (normalised)")
for i, ax in enumerate(axes[1:], start=1):
    aug = spatial_augment(fixed, aug_rng)
    plot_skeleton(ax, aug[32], f"spatial_augment, draw {i}")
plt.show()

# Temporal augmentation changes the frame count (it runs BEFORE the 64-frame resample):
for _ in range(5):
    out = temporal_augment(clean, aug_rng)
    print(f"temporal_augment: {clean.shape[0]} frames -> {out.shape[0]} frames")


## 8 — Models: from-scratch transformer vs matched LSTM baseline

CLI: `python -m src.models` (prints parameter counts)

`src/models.py` implements the **pre-LN transformer encoder from scratch** (manual
scaled-dot-product attention — no `nn.Transformer` / `nn.MultiheadAttention`): input
projection → learnable CLS token → learned positional embeddings → 4 encoder layers
(d_model 192, 6 heads, FFN 384) → LayerNorm → linear head on the CLS position. The
baseline is a unidirectional 2×256 LSTM whose final hidden state feeds the same kind of
head. Both map `(B, 64, 315)` to 50-class logits, at deliberately matched capacity (~1M
parameters).

In [ ]:
from src.models import build_model, count_parameters

transformer = build_model({"arch": "transformer"})
lstm = build_model({"arch": "lstm"})

t_params = count_parameters(transformer)
l_params = count_parameters(lstm)
print(f"TransformerClassifier: {t_params:,} ({t_params / 1e6:.2f}M)")
print(f"LSTMClassifier:        {l_params:,} ({l_params / 1e6:.2f}M)")
print(f"relative difference |T-L|/T: {abs(t_params - l_params) / t_params:.3f}")

# Shape check: a dummy batch through both models.
dummy = torch.zeros(2, 64, 315)
with torch.inference_mode():
    print(f"\ntransformer(dummy) -> {tuple(transformer(dummy).shape)}")
    print(f"lstm(dummy)        -> {tuple(lstm(dummy).shape)}")


## 9 — The 12 experiment configs

CLI: `python configs/gen_configs.py`

Every experiment is one YAML file: {transformer, lstm} × {aug, noaug} × seeds {42, 43, 44}.
All hyperparameters (80 epochs, batch 32, AdamW lr 3e-4, weight decay 0.01, 5 warm-up
epochs then cosine decay, label smoothing 0.1) are identical across the grid — only
architecture, augmentation flag and seed vary. The generator rewrites all 12 files, so individual
files should not be edited by hand; changes belong in the template in
`configs/gen_configs.py`.

In [ ]:
%run configs/gen_configs.py


In [ ]:
cfg = yaml.safe_load(Path("configs/transformer_aug_s42.yaml").read_text(encoding="utf-8"))
cfg


# Part C — Training: the full experimental grid, fresh

Three groups of runs, exactly as the dissertation describes, all trained live here:

1. **v1 12-run grid** — {transformer, lstm} x {aug, noaug} x seeds {42, 43, 44} on the
   organisation-grouped v1 split (`configs/*.yaml` -> `results/runs/`).
2. **lr / weight-decay selection grid** — {3e-4, 1e-3} x {0.01, 0.05} per architecture,
   validation-only (`results/lr_grid/`), reproducing the hyperparameter freeze.
3. **v2 12-run grid** (Part D) — the grid the final results rest on.

`run_training` seeds python/numpy/torch and forces deterministic cuDNN; each run
directory records its exact config and library versions (`env.txt`). Training uses the
GPU automatically when available. The dissertation-era v1 records (recorded
2026-07-09) came from CPU training under torch 2.3.1, so small deviations in this fresh
retrain are expected and honest - the v2 grid in Part D is the one that must (and does)
reproduce the reported final results.

## 10 — Sanity checks (run before trusting any result)

CLI: `python -m src.train --config configs/transformer_aug_s42.yaml --overfit-batch`
and `... --shuffle-labels`

- **Overfit one batch** — trains on a single fixed batch with weight decay disabled and
  *asserts* ≥ 95% batch accuracy within 200 steps. Passing proves the model / loss /
  optimiser wiring can learn at all. Writes no files.
- **Label shuffle** — permutes the training labels once (seeded) and trains normally;
  validation accuracy must stay near chance (1/50 = 2%). If it doesn't, labels are leaking
  into the features. The run is written under `{run_id}_shufflelabels` so it never
  clobbers a real run. This is a full training run, hence off by default here.

In [ ]:
from src.train import run_training

run_training(dict(cfg), overfit_batch=True)  # asserts >=95% within 200 steps


In [ ]:
RUN_SHUFFLE_LABELS = True  # full 80-epoch run; writes results/runs/transformer_aug_s42_shufflelabels

if RUN_SHUFFLE_LABELS:
    run_training(dict(cfg), shuffle_labels=True)
else:
    print("skipped (RUN_SHUFFLE_LABELS = True)")


In [ ]:
# The full 12-run grid — equivalent of looping `python -m src.train --config` over
# configs/*.yaml. Hours on CPU, and it OVERWRITES the finished runs in results/runs/.
RUN_FULL_GRID = True

if RUN_FULL_GRID:
    for cfg_path in sorted(Path("configs").glob("*.yaml")):
        print(f"=== {cfg_path.name} ===")
        run_training(yaml.safe_load(cfg_path.read_text(encoding="utf-8")))
else:
    done = sorted(p.parent.name for p in Path("results/runs").glob("*/best.pt"))
    print(f"skipped (RUN_FULL_GRID = True) - {len(done)} finished runs on disk:")
    for name in done:
        print(" ", name)


In [ ]:
# Training curves of a finished run, straight from its metrics.csv.
metrics_csv = Path("results/runs/transformer_aug_s42/metrics.csv")
m = pd.read_csv(metrics_csv)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(m["epoch"], m["train_loss"], label="train")
axes[0].plot(m["epoch"], m["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[1].plot(m["epoch"], m["train_acc"], label="train")
axes[1].plot(m["epoch"], m["val_acc"], label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
fig.suptitle(metrics_csv.parent.name)
plt.show()

print(f"best val acc: {m['val_acc'].max():.4f} at epoch {m['val_acc'].idxmax() + 1}")


In [ ]:
# Evaluate every v1 run on validation - Part E's dissertation tables read these
# eval_val outputs (predictions, per-class accuracy, confusion matrices).
from src.evaluate import run_evaluation as _run_eval

for run_dir in sorted(Path("results/runs").glob("*")):
    if not (run_dir / "best.pt").is_file() or run_dir.name.endswith("_shufflelabels"):
        continue
    print(f"--- {run_dir.name}")
    _run_eval(str(run_dir / "best.pt"), "val")


## 12 — Hyperparameter selection: the lr / weight-decay grid

Script: `python scripts/run_lr_grid.py`

Before the 12-run grid was launched, learning rate and weight decay were chosen on
validation only: {3e-4, 1e-3} × {0.01, 0.05}, one run per architecture, seed 42 with
augmentation, written under `results/lr_grid/`. The winners were frozen into the
experiment configs. The test set is never touched here. This is 8 full training runs,
so it is off by default — the summary of the finished runs on disk is shown instead.

In [ ]:
RUN_LR_GRID = True  # 8 x 80-epoch runs

if RUN_LR_GRID:
    for arch in ("transformer", "lstm"):
        base = yaml.safe_load(
            Path(f"configs/{arch}_aug_s42.yaml").read_text(encoding="utf-8"))
        for lr in (3e-4, 1e-3):
            for wd in (0.01, 0.05):
                grid_cfg = dict(base)
                grid_cfg.update(lr=lr, weight_decay=wd,
                                run_id=f"{arch}_lr{lr:g}_wd{wd:g}",
                                out_dir="results/lr_grid")
                out = run_training(grid_cfg)
                print(f"[lr-grid] {arch} lr={lr:g} wd={wd:g} "
                      f"-> best val acc {out['best_val_acc']:.4f}")
else:
    rows = []
    for run_dir in sorted(Path("results/lr_grid").glob("*")):
        mcsv = run_dir / "metrics.csv"
        if mcsv.exists():
            rows.append({"run": run_dir.name,
                         "best_val_acc": pd.read_csv(mcsv)["val_acc"].max()})
    if rows:
        display(pd.DataFrame(rows).sort_values("best_val_acc", ascending=False))
    else:
        print("no finished lr-grid runs found under results/lr_grid/")


## 13 — Evaluation, run comparison, and McNemar's test

CLI: `python -m src.evaluate --checkpoint results/runs/<run>/best.pt --split val`

`run_evaluation` loads a checkpoint (architecture and label list travel inside `best.pt`),
runs the validation clips through the exact same preprocessing as training (no
augmentation), and writes four artefacts into `{run_dir}/eval_val/`: `predictions.csv`
(with top-5 columns), `per_class_accuracy.csv`, `confusion_matrix.csv` and
`confusion_matrix.png`. It reports top-1 / top-5 accuracy and macro-F1.

In [ ]:
from src.evaluate import run_evaluation

eval_metrics = run_evaluation("results/runs/transformer_aug_s42/best.pt", "val")


In [ ]:
from IPython.display import Image, display

out_dir = Path(eval_metrics["out_dir"])
display(Image(filename=str(out_dir / "confusion_matrix.png"), width=700))

per_class = pd.read_csv(out_dir / "per_class_accuracy.csv")
print("hardest classes on validation:")
per_class.sort_values("accuracy").head(10)


In [ ]:
# Best validation accuracy of every finished run, as mean +- std per condition
# (the dissertation reports results over seeds 42/43/44).
rows = []
for run_dir in sorted(Path("results/runs").glob("*")):
    mcsv = run_dir / "metrics.csv"
    parts = run_dir.name.rsplit("_", 2)
    if not mcsv.exists() or len(parts) != 3 or run_dir.name.endswith("_shufflelabels"):
        continue
    arch, aug, seed = parts
    rows.append({"run": run_dir.name, "arch": arch, "augmentation": aug, "seed": seed,
                 "best_val_acc": pd.read_csv(mcsv)["val_acc"].max()})
runs_df = pd.DataFrame(rows)
display(runs_df.sort_values("best_val_acc", ascending=False))
runs_df.groupby(["arch", "augmentation"])["best_val_acc"].agg(["mean", "std", "count"])


In [ ]:
# McNemar's test: are two models' error patterns significantly different on the SAME clips?
# CLI: python -m src.evaluate --mcnemar runA/eval_val/predictions.csv runB/eval_val/predictions.csv
from src.evaluate import run_mcnemar

pair = ("transformer_aug_s42", "lstm_aug_s42")
for run in pair:  # make sure both runs have validation predictions
    if not Path(f"results/runs/{run}/eval_val/predictions.csv").exists():
        run_evaluation(f"results/runs/{run}/best.pt", "val")

mcnemar_result = run_mcnemar(
    f"results/runs/{pair[0]}/eval_val/predictions.csv",
    f"results/runs/{pair[1]}/eval_val/predictions.csv",
)


## 14 — Latency benchmark

CLI: `python -m src.benchmark --checkpoint ... --synthetic --frames 1000 --stride 8`
(or `--webcam 0` / `--video clip.mp4` for the full pipeline including MediaPipe)

Times every stage (capture, holistic, normalise, forward) with `perf_counter`, reports
medians and 95th percentiles after warm-up, and checks the real-time criteria:
**≥ 15 fps end-to-end** and **< 100 ms median per-window classification**. One row per
run is appended to `results/latency.csv`. `--synthetic` feeds random landmark frames so
it runs without a camera or mediapipe — the numbers then cover normalise + forward only.

In [ ]:
from src.benchmark import run_benchmark

bench_row = run_benchmark("results/runs/transformer_aug_s42/best.pt",
                          mode="synthetic", frames=300, stride=8)


In [ ]:
# Full-pipeline benchmark against the webcam (needs a camera + mediapipe):
RUN_WEBCAM_BENCH = False

if RUN_WEBCAM_BENCH:
    run_benchmark("results/runs/transformer_aug_s42/best.pt",
                  mode="webcam", source=0, frames=1000, stride=8)
else:
    print("skipped (RUN_WEBCAM_BENCH = False)")


# Part D — v2: the re-sourced held-out evaluation, fresh

## 18 — v2 partition: the re-sourced held-out evaluation (2026-07-27)

The originally planned self-recorded test set was replaced on
2026-07-27 by a **two-part re-sourced held-out evaluation**:

1. **Organisation-held-out one-shot test** — the three v1 validation organisations
   (*corpusngt, deafway, gpnhs*; 69 clips, 49/50 classes), which no model was ever
   trained on, become the test partition. Model selection moves to a new clip-stratified
   validation split (exactly one clip per word, seeded) inside the eleven training
   organisations, and the 12-run grid is **re-trained** against it
   (`data/splits_v2.json`, `configs_v2/`, `results/runs_v2/`,
   derivation: `src/make_v2_partition.py`), so no selected model ever used test clips
   for selection. Organisation grouping is the methodology's signer proxy, so this
   preserves source/signer-independent evaluation.
2. **BSL SignBank cross-corpus one-shot test** — 100 citation-form clips of Deaf signers
   from an entirely unrelated corpus (next subsection).

Disclosed caveats: learning rate and weight decay were frozen
in the Chapter-3 grid using the former validation organisations, so the promoted test
partition influenced *hyperparameter* (never model-weight) selection; *water* has no
clips in partition 1 and is covered by partition 2 only; the v2 validation split shares
organisations with train and is a model-selection split, not a generalisation estimate.

**GPU note:** `run_training` auto-selects CUDA when available — the v2 grid takes
~10 minutes on a laptop GPU (exact library versions per run are in each run's `env.txt`).
Sanity gates: the overfit-one-batch check (run once, against the original partition — it tests the training-loop wiring, which the re-partition does not change) passed at 0.969 within 24 steps; the label-shuffle gate was re-run against the v2 split and stayed at chance — val acc **0.020** (`results/runs_v2/transformer_aug_s42_shufflelabels`).

In [ ]:
# Build the v2 partition FRESH (splits_v2.json, data/test_landmarks/, configs_v2/),
# then verify the regenerated split matches the committed record byte-for-byte.
committed_v2 = json.loads(Path("data/splits_v2.json").read_text(encoding="utf-8"))
from src.make_v2_partition import main as make_v2_main

make_v2_main()
regenerated_v2 = json.loads(Path("data/splits_v2.json").read_text(encoding="utf-8"))
assert regenerated_v2 == committed_v2, \
    "regenerated v2 partition differs from the committed data/splits_v2.json!"
print("\nVERIFIED: regenerated splits_v2.json matches the committed partition exactly")


In [ ]:
RUN_FULL_GRID_V2 = True  # ~10 min on GPU / ~1 h on CPU; OVERWRITES results/runs_v2/

if RUN_FULL_GRID_V2:
    for cfg_path in sorted(Path("configs_v2").glob("*.yaml")):
        print(f"=== {cfg_path.name} ===")
        run_training(yaml.safe_load(cfg_path.read_text(encoding="utf-8")))
else:
    done = sorted(p.parent.name for p in Path("results/runs_v2").glob("*/best.pt"))
    print(f"skipped (RUN_FULL_GRID_V2 = True) - {len(done)} finished v2 runs on disk:")
    for name in done:
        print(" ", name)

In [ ]:
# Per-run best val acc, per-group mean±std, and the frozen per-group selection.
# Writes results/validation_summary_v2.csv.
%run scripts/summarise_v2.py

### The SignBank cross-corpus test set

CLI: `python -m src.download_signbank --manifest data/signbank_gloss_map_curated.csv`

100 isolated citation-form clips (all 50 words, up to 3 regional variants each) from
**UCL BSL SignBank** (bslsignbank.ucl.ac.uk) — a corpus of Deaf signers entirely disjoint
from the 14 SignBSL contributing organisations, downloaded for **non-commercial
academic research** and used solely as a held-out evaluation
set, never redistributed.

Provenance chain, all committed:
- `data/signbank_gloss_map.csv` — the full 103-row word→gloss→video mapping built from
  SignBank's search UI, with per-row match confidence and sense notes;
- `data/signbank_gloss_map_curated.csv` — 100 rows after curation: no video file may
  serve two labels (WAVE-HAND stays with *goodbye*, WHAT with *what*) and one
  unverifiable entry (*ask* → gloss TOUCH) removed;
- `data/signbank_metadata.csv` — per-file download record (URL, date, bytes).

Four words carry a **sense flag** (SignBank has no same-gloss entry): *stop*=FINISH,
*today*=NOW, *need*=WANT, *goodbye*=WAVE-HAND — results are reported with and without
them (see the subset analysis below). Extraction reuses section 2's pipeline; hand
dropout on this studio material is only ~2.3%.

In [ ]:
RUN_SIGNBANK_DOWNLOAD = False  # polite sequential downloads; for non-commercial academic research use
RUN_SIGNBANK_EXTRACT = True   # mediapipe 0.10.9; ~1.5 s per clip

if RUN_SIGNBANK_DOWNLOAD:
    !{sys.executable} -m src.download_signbank --manifest data/signbank_gloss_map_curated.csv --max-per-word 3
if RUN_SIGNBANK_EXTRACT:
    !{sys.executable} -m src.extract --videos data/test_videos_signbank --out data/test_landmarks_signbank

n_mp4 = len(list(Path("data/test_videos_signbank").glob("*/*.mp4")))
n_npz = len(list(Path("data/test_landmarks_signbank").glob("*/*.npz")))
print(f"on disk: {n_mp4} SignBank mp4s, {n_npz} extracted landmark files")

### One-shot test — reproduction of the recorded 2026-07-27 evaluation

The protocol's one-shot test evaluations were performed **once**, on 2026-07-27, after
model selection was frozen on the v2 validation split; the results were recorded on
that date. The cell below **reproduces** those recorded evaluations from the
grid just retrained. It makes no new selection decision: the four evaluated checkpoints
are the recorded frozen selections (best val accuracy per group, all seed 42), and the
recorded top-1 values are printed beside each reproduced number. On the original
machine and library versions the reproduction is exact; on different hardware small
deviations are expected.

In [ ]:
import shutil

from src.evaluate import run_evaluation, run_mcnemar

SELECTED = ["transformer_noaug_s42", "transformer_aug_s42", "lstm_aug_s42", "lstm_noaug_s42"]
RECORDED = {  # recorded one-shot results, 2026-07-27
    "orgheldout": {"transformer_noaug_s42": 0.1304, "transformer_aug_s42": 0.0870,
                   "lstm_aug_s42": 0.0145, "lstm_noaug_s42": 0.0145},
    "signbank": {"transformer_noaug_s42": 0.1400, "transformer_aug_s42": 0.1400,
                 "lstm_aug_s42": 0.0600, "lstm_noaug_s42": 0.0500},
}
rows = []
for run in SELECTED:
    ckpt = f"results/runs_v2/{run}/best.pt"
    for part, lm_dir in [("orgheldout", "data/test_landmarks"),
                         ("signbank", "data/test_landmarks_signbank")]:
        print(f"=== {run} / {part}")
        m = run_evaluation(ckpt, "test", landmarks_dir=lm_dir)
        out, dst = (Path(f"results/runs_v2/{run}/eval_test"),
                    Path(f"results/runs_v2/{run}/eval_test_{part}"))
        if dst.exists():
            shutil.rmtree(dst)
        out.rename(dst)
        rows.append({"run": run, "test_set": part, "n": m["n"],
                     "top1_this_run": round(m["top1"], 4),
                     "top1_recorded": RECORDED[part][run],
                     "top5": round(m["top5"], 4), "macro_f1": round(m["macro_f1"], 4)})
display(pd.DataFrame(rows))

print("\nMcNemar, organisation-held-out (recorded exact p = 0.0215):")
run_mcnemar("results/runs_v2/transformer_noaug_s42/eval_test_orgheldout/predictions.csv",
            "results/runs_v2/lstm_aug_s42/eval_test_orgheldout/predictions.csv")
print("\nMcNemar, SignBank cross-corpus (recorded exact p = 0.0215):")
run_mcnemar("results/runs_v2/transformer_noaug_s42/eval_test_signbank/predictions.csv",
            "results/runs_v2/lstm_aug_s42/eval_test_signbank/predictions.csv")


In [ ]:
# Sense-variant analysis: overall vs exact-keyword-match (high-confidence) subset,
# and excluding the four words where SignBank has no same-gloss entry.
%run scripts/signbank_subset.py

# Part E — Dissertation figures and tables

*(merged from `notebooks/results.ipynb`; reads the v1 grid trained in Part C)*

# Results — dissertation figures and tables

Regenerates every Chapter 5 figure/table from `results/runs/*` and
`results/latency.csv`. Run after the 12-run grid has been trained and each
checkpoint evaluated with `python -m src.evaluate ... --split val`
(README workflow steps 6-9).

Run groups: **A** transformer+aug, **B** transformer no-aug, **C** LSTM+aug,
**D** LSTM no-aug; three seeds (42/43/44) each.

In [ ]:
from pathlib import Path
import sys
import subprocess

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

RUNS = REPO / 'results' / 'runs'
if not RUNS.exists():
    raise FileNotFoundError(
        f'{RUNS} not found - train the 12-run grid first (README workflow step 6).')

run_dirs = sorted(d for d in RUNS.iterdir() if (d / 'metrics.csv').exists()
                  and not d.name.endswith('_shufflelabels'))
print(f'{len(run_dirs)} runs found: {[d.name for d in run_dirs]}')


def parse_run(name: str):
    arch, aug, seed = name.rsplit('_', 2)
    return arch, aug == 'aug', int(seed.lstrip('s'))

## Learning curves (Figure for §5.1 Training behaviour)

Validation accuracy per epoch for all runs; solid = augmented, dashed = no
augmentation. Destination: Chapter 5, §5.1.

In [ ]:
records = []
curves = {}
for d in run_dirs:
    m = pd.read_csv(d / 'metrics.csv')
    arch, aug, seed = parse_run(d.name)
    curves[d.name] = m
    records.append({'run_id': d.name, 'arch': arch, 'augment': aug, 'seed': seed,
                    'best_val_acc': m['val_acc'].max(),
                    'best_epoch': int(m.loc[m['val_acc'].idxmax(), 'epoch'])})
runs = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for name, m in curves.items():
    arch, aug, seed = parse_run(name)
    ax = axes[0] if arch == 'transformer' else axes[1]
    ax.plot(m['epoch'], m['val_acc'], alpha=0.8,
            ls='-' if aug else '--', label=name)
for ax, title in zip(axes, ['transformer', 'lstm']):
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.legend(fontsize=7)
axes[0].set_ylabel('val accuracy')
fig.suptitle('Validation accuracy over 80 epochs')
plt.tight_layout()
plt.show()
runs.sort_values(['arch', 'augment', 'seed'])

## Mean ± std over seeds per run group (Table for §5.1)

Best validation accuracy aggregated over seeds 42/43/44 for groups A-D.
Destination: Chapter 5, §5.1 (headline results table).

In [ ]:
GROUPS = {'A': ('transformer', True), 'B': ('transformer', False),
          'C': ('lstm', True), 'D': ('lstm', False)}
rows = []
for g, (arch, aug) in GROUPS.items():
    sel = runs[(runs['arch'] == arch) & (runs['augment'] == aug)]['best_val_acc']
    rows.append({'group': g, 'arch': arch, 'augment': aug, 'n_seeds': len(sel),
                 'val_acc': f'{sel.mean():.3f} +/- {sel.std(ddof=1):.3f}'
                            if len(sel) > 1 else f'{sel.mean():.3f}'})
group_table = pd.DataFrame(rows).set_index('group')
group_table

## RQ1 — Transformer vs LSTM (Table for §5.2)

Top-1 / top-5 / macro-F1 recomputed from each run's `eval_val/predictions.csv`,
aggregated over seeds for the augmented condition. Destination: Chapter 5, §5.2
(architecture comparison, RQ1).

In [ ]:
def eval_metrics(run_dir: Path, split: str = 'val'):
    p = run_dir / f'eval_{split}' / 'predictions.csv'
    if not p.exists():
        return None
    df = pd.read_csv(p)
    top5_cols = [c for c in df.columns if c.startswith('top5')]
    top5 = (df.apply(lambda r: r['true'] in [r[c] for c in top5_cols], axis=1).mean()
            if top5_cols else np.nan)
    return {'top1': df['correct'].mean(), 'top5': top5,
            'macro_f1': f1_score(df['true'], df['pred'], average='macro'),
            'predictions': p}


rows = []
for _, r in runs.iterrows():
    m = eval_metrics(RUNS / r['run_id'])
    if m is not None:
        rows.append({'run_id': r['run_id'], 'arch': r['arch'],
                     'augment': r['augment'], 'seed': r['seed'],
                     'top1': m['top1'], 'top5': m['top5'],
                     'macro_f1': m['macro_f1']})
ev = pd.DataFrame(rows)
if ev.empty:
    print('No eval_val outputs found - run `python -m src.evaluate '
          '--checkpoint results/runs/<run>/best.pt --split val` per run first.')
else:
    rq1 = (ev[ev['augment']]
           .groupby('arch')[['top1', 'top5', 'macro_f1']]
           .agg(['mean', 'std']).round(3))
    display(rq1)

## RQ2 — Augmentation ablation deltas (Table for §5.3)

Mean top-1 with vs without augmentation per architecture, and the delta.
Destination: Chapter 5, §5.3 (augmentation ablation, RQ2).

In [ ]:
if not ev.empty:
    pivot = ev.groupby(['arch', 'augment'])['top1'].mean().unstack()
    pivot.columns = ['noaug', 'aug']
    pivot['delta_aug'] = pivot['aug'] - pivot['noaug']
    display(pivot.round(3))

## Confusion matrices — best transformer and best LSTM (Figures for §5.2)

Loaded from each best run's `eval_val/confusion_matrix.csv`. Destination:
Chapter 5, §5.2 (error analysis).

In [ ]:
def best_run(arch: str):
    if ev.empty:
        return None
    sel = ev[(ev['arch'] == arch) & (ev['augment'])]
    if sel.empty:
        sel = ev[ev['arch'] == arch]
    return None if sel.empty else sel.loc[sel['top1'].idxmax(), 'run_id']


for arch in ['transformer', 'lstm']:
    rid = best_run(arch)
    if rid is None:
        print(f'no evaluated {arch} runs')
        continue
    cm_path = RUNS / rid / 'eval_val' / 'confusion_matrix.csv'
    if not cm_path.exists():
        print(f'{cm_path} missing')
        continue
    cm = pd.read_csv(cm_path, index_col=0)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm.values, cmap='Blues')
    ax.set_title(f'Confusion matrix - {rid} (val)')
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

## Latency (Table for §5.4)

Per-stage median/p95 and end-to-end fps from `results/latency.csv`
(written by `python -m src.benchmark`). Destination: Chapter 5, §5.4
(real-time criterion: >= 15 fps, < 100 ms per window).

In [ ]:
lat_path = REPO / 'results' / 'latency.csv'
if lat_path.exists():
    display(pd.read_csv(lat_path))
else:
    print('results/latency.csv not found - run `python -m src.benchmark` first '
          '(README workflow step 9).')

## McNemar's test — best transformer vs best LSTM (§5.2)

Exact binomial McNemar on paired per-clip correctness (alpha = 0.05), via the
project CLI so the dissertation quotes exactly what the tooling reports.
Destination: Chapter 5, §5.2 (significance of the architecture difference).

In [ ]:
best_t, best_l = best_run('transformer'), best_run('lstm')
if best_t and best_l:
    preds_a = RUNS / best_t / 'eval_val' / 'predictions.csv'
    preds_b = RUNS / best_l / 'eval_val' / 'predictions.csv'
    result = subprocess.run(
        [sys.executable, '-m', 'src.evaluate', '--mcnemar',
         str(preds_a), str(preds_b)],
        cwd=str(REPO), capture_output=True, text=True)
    print(f'A = {best_t}\nB = {best_l}\n')
    print(result.stdout or result.stderr)
else:
    print('need at least one evaluated transformer and one evaluated lstm run')

# Optional live demos

The real-time webcam demo and the Gradio app need a camera and an interactive session, so they stay behind flags - flip them live if wanted.

## 15 — Real-time webcam demo

CLI: `python -m src.realtime --checkpoint results/runs/<run>/best.pt`

The live loop keeps a rolling buffer of the last **64 normalised frames**, classifies
every **8 frames**, and only displays a prediction once the same top-1 class has cleared a
**0.6 confidence threshold for 3 consecutive windows** (the `PredictionSmoother`). It
opens a native OpenCV window, so it works from a *local* Jupyter server but not from a
remote/hosted one — press `q` in the window to quit.

The smoother is worth understanding on its own, so the first cell drives it with a
scripted confidence sequence.

In [ ]:
from src.realtime import PredictionSmoother

smoother = PredictionSmoother(threshold=0.6, consecutive=3)
scripted = [(4, 0.9), (4, 0.8), (7, 0.7), (7, 0.9), (7, 0.85),  # 7 confirmed on the 3rd
            (2, 0.4), (2, 0.95), (2, 0.9)]                       # low conf resets the streak
for cls, conf in scripted:
    confirmed, c = smoother.update(cls, conf)
    shown = f"class {confirmed} ({c:.2f})" if confirmed is not None else "listening..."
    print(f"window top-1 = class {cls} @ {conf:.2f}  ->  displayed: {shown}")


In [ ]:
RUN_REALTIME = False  # opens an OpenCV window; local Jupyter only; press q to quit

if RUN_REALTIME:
    from src.realtime import run as realtime_run

    realtime_run("results/runs/transformer_aug_s42/best.pt",
                 camera=0, stride=8, threshold=0.6, consecutive=3)
else:
    print("skipped (RUN_REALTIME = False)")


## 16 — Gradio demo app

CLI: `python app/app.py`

Upload or webcam-record a 2–3 second clip of a single sign; the app runs MediaPipe
extraction plus the shared preprocessing pipeline and shows the top-5 predictions.
The checkpoint is resolved from the `BSL_CHECKPOINT` env var, then `app/model/best.pt`,
then the newest `results/runs/*/best.pt`. Launched from a notebook, Gradio renders the
interface inline. It is a research demonstrator for learning and lookup — **not an
interpreting service**.

In [ ]:
RUN_APP = False  # needs gradio + mediapipe; serves on a local port until stopped

if RUN_APP:
    import importlib.util

    spec = importlib.util.spec_from_file_location("bsl_app",
                                                  str(REPO_ROOT / "app" / "app.py"))
    bsl_app = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(bsl_app)  # builds `demo` at import time
    bsl_app.demo.queue(max_size=8).launch(inline=True)
    # later: bsl_app.demo.close()
else:
    print("skipped (RUN_APP = False)")


## 17 — Test suite

CLI: `pytest`

The pytest suite covers the model shapes and parameter counts, normalisation invariances,
augmentation bounds, dataset/resampling behaviour, evaluation metrics and a training smoke
test — and it deliberately runs **without any data downloaded and without mediapipe**, so
it works on any machine as a first sanity check.

In [ ]:
!{sys.executable} -m pytest -q


# Part F — Recap: everything this run produced

Key headline numbers from THIS run next to the recorded values, then an inventory of
every artefact the notebook just created from the shipped videos alone.

In [ ]:
summary = pd.read_csv("results/validation_summary_v2.csv")
bt_org = pd.read_csv("results/runs_v2/transformer_noaug_s42/eval_test_orgheldout/predictions.csv")
bt_sb = pd.read_csv("results/runs_v2/transformer_noaug_s42/eval_test_signbank/predictions.csv")
headline = [
    ("v2 val, best transformer", summary.loc[summary.run_id == "transformer_noaug_s42",
                                             "best_val_acc"].iloc[0], 0.200),
    ("org-held-out test top-1, best transformer", bt_org["correct"].mean(), 0.1304),
    ("SignBank test top-1, best transformer", bt_sb["correct"].mean(), 0.1400),
]
print(f"{'headline number':<44} {'this run':>9} {'recorded':>9}")
for name, got, rec in headline:
    print(f"{name:<44} {got:>9.4f} {rec:>9.4f}")

print("\nArtefacts produced by this run:")
for p in ["data/landmarks", "data/test_landmarks", "data/test_landmarks_signbank",
          "results/runs", "results/lr_grid", "results/runs_v2",
          "results/latency.csv", "results/validation_summary_v2.csv"]:
    q = Path(p)
    n = sum(1 for x in q.rglob("*") if x.is_file()) if q.is_dir() else int(q.exists())
    print(f"  {p:<38} {n:>5} files")
